# Phase 11: Manufacturing Copilot

This notebook documents how the project stores model outputs in SQLite and serves them through a Streamlit application. The copilot uses the production-safe Phase 6 LightGBM score as the official failure probability. Phase 7 SHAP, Phase 8 process mining, Phase 9 graph analytics, and Phase 10 advanced-AI outputs are supporting diagnostic evidence.

## 1. Design and governance

- **Database:** SQLite provides a portable, queryable store without requiring a server.
- **Application:** Streamlit provides KPI summaries, charts, filters, drill-down tables, and question answering.
- **Natural language:** Questions are mapped to reviewed intents and parameterized SQL. This makes answers reproducible and prevents unrestricted SQL generation.
- **Root-cause language:** SHAP and graph outputs describe model associations and investigation priorities, not proven physical causation.
- **Deployment boundary:** This use case uses historical competition data. A real factory implementation requires live data pipelines, access control, monitoring, and engineering validation.

In [ ]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATABASE_PATH = PROJECT_ROOT / 'data' / 'database' / 'manufacturing_copilot.db'
DATABASE_PATH

## 2. Build the database

The build script validates required Phase 3 and Phase 6-10 outputs, recalculates official validation probabilities from the saved Phase 6 model, joins available advanced-AI diagnostics, creates summary metrics, and adds indexes for product lookup and risk ranking.

In [ ]:
from src.data.phase11_manufacturing_copilot import build_database, write_report

row_counts = build_database()
write_report(row_counts)
pd.Series(row_counts, name='rows').sort_values(ascending=False)

## 3. Inspect database tables

The source catalog records the file or derivation used for each table. This provides a simple audit trail for the application.

In [ ]:
connection = sqlite3.connect(DATABASE_PATH)
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection
)
display(tables)
display(pd.read_sql_query('SELECT * FROM source_catalog ORDER BY table_name', connection))

## 4. Production performance summary

The summary table contains the metrics used by the first application view. Validation metrics describe the held-out Phase 6 validation population. Test predictions describe an unlabeled production-like population and therefore cannot be treated as measured failures.

In [ ]:
summary = pd.read_sql_query(
    'SELECT metric, value, unit, interpretation FROM production_summary ORDER BY display_order',
    connection,
)
display(summary)

## 5. Review high-risk products

The official failure probability comes from Phase 6. Advanced-AI fields are supporting signals and may be missing for test rows outside the Phase 10 preview.

In [ ]:
high_risk = pd.read_sql_query(
    '''
    SELECT Id, failure_probability, predicted_failure,
           isolation_forest_anomaly_score, trajectory_failure_risk
    FROM product_predictions
    WHERE split='test'
    ORDER BY failure_probability DESC
    LIMIT 20
    ''',
    connection,
)
display(high_risk)

## 6. Natural-language querying

The query engine recognizes manufacturing intents such as failure rate, bottleneck, station root cause, critical graph node, process route, model validation, and production summary. Station identifiers such as `L3_S32` are extracted and passed to parameterized SQL.

In [ ]:
from src.copilot.query_engine import answer_question

questions = [
    'Which stations have the highest failure rate?',
    'What are the top bottlenecks?',
    'Why is L3_S32 risky?',
    'How good is the production-safe model?',
]

for question in questions:
    answer = answer_question(connection, question)
    print(f'QUESTION: {question}')
    print(f'ANSWER: {answer.narrative}\n')
    display(answer.data.head(10))

## 7. Run the Streamlit application

Run this command from the project root:

```powershell
.\.venv\Scripts\streamlit.exe run app\phase11_manufacturing_copilot.py
```

The app contains five views: Overview, Risk Monitor, Root Causes, Process Intelligence, and Copilot.

## 8. Production implementation notes

For a future manufacturing client, replace CSV refreshes with governed batch or streaming ingestion, add identity-based access, log every model and copilot answer, monitor feature and prediction drift, connect alerts to engineering workflow, and retrain only after labeled outcomes are available. Keep the reviewed SQL intent layer for operational questions; an optional language model can later translate broader wording into these approved intents without receiving unrestricted database access.

In [ ]:
connection.close()